# 🏏 IPL Cricket Stats Analysis
### Data Analysis using Pandas, NumPy, Matplotlib & Seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

## 1. Load Dataset

> 📥 Dataset: [IPL Complete Dataset on Kaggle](https://www.kaggle.com/datasets/patrickb1912/ipl-complete-dataset-20082020)
>
> Place **`matches.csv`** and **`deliveries.csv`** in the same folder as this notebook.
>
> **matches.csv columns:**  
> `id, season, city, date, match_type, player_of_match, venue, team1, team2,`  
> `toss_winner, toss_decision, winner, result, result_margin, target_runs,`  
> `target_overs, super_over, method, umpire1, umpire2`
>
> **deliveries.csv columns:**  
> `match_id, inning, batting_team, bowling_team, over, ball, batter, bowler,`  
> `non_striker, batsman_runs, extra_runs, total_runs, extras_type,`  
> `is_wicket, player_dismissed, dismissal_kind, fielder`

In [ ]:
matches    = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

print('Matches Dataset:')
display(matches.head())
print('\nDeliveries Dataset:')
display(deliveries.head())

## 2. Dataset Overview

In [ ]:
print('== MATCHES DATASET ==')
print(f'Shape   : {matches.shape}')
print(f'Columns : {matches.columns.tolist()}')
print('\nData Types:')
print(matches.dtypes)
print(f'\nMissing Values:\n{matches.isnull().sum()}')

In [ ]:
print('== DELIVERIES DATASET ==')
print(f'Shape   : {deliveries.shape}')
print(f'Columns : {deliveries.columns.tolist()}')
print('\nData Types:')
print(deliveries.dtypes)
print(f'\nMissing Values:\n{deliveries.isnull().sum()}')

In [ ]:
matches.describe(include='all')

## 3. Data Cleaning

Key schema facts confirmed from the data:
- `matches`: no `win_by_runs`/`win_by_wickets` — result is stored in `result` (values: `runs`, `wickets`, `tie`, `no result`) + `result_margin`
- `matches`: no `umpire3`; umpires are `umpire1`, `umpire2` only
- `matches`: `season` is stored as `'2007/08'` strings → extract 4-digit year
- `deliveries`: batter column is called `batter` (not `batsman`)
- `deliveries`: `over` is 0-indexed (0–19, not 1–20) → phase boundaries adjust accordingly
- `deliveries`: `is_wicket` is an integer flag (0/1), not `dismissal_kind`
- `deliveries`: `player_dismissed`, `dismissal_kind`, `fielder` use NaN (not a string sentinel)

In [ ]:
print('Before cleaning (matches):')
print(matches.isnull().sum())

# ── matches.csv ──────────────────────────────────────────────────────────────
matches['city']            = matches['city'].fillna('Unknown')
matches['winner']          = matches['winner'].fillna('No Result')
matches['player_of_match'] = matches['player_of_match'].fillna('N/A')
matches['method']          = matches['method'].fillna('Normal')   # NaN = no D/L
matches['result_margin']   = matches['result_margin'].fillna(0)

# Extract 4-digit year from strings like '2007/08' or plain '2010'
matches['season'] = (
    matches['season'].astype(str)
    .str.extract(r'(\d{4})')[0]
    .astype(int)
)

# Standardise franchise name changes across seasons
team_rename = {
    'Delhi Daredevils'           : 'Delhi Capitals',
    'Deccan Chargers'            : 'Sunrisers Hyderabad',
    'Rising Pune Supergiant'     : 'Rising Pune Supergiants',
    'Kings XI Punjab'            : 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
}
for col in ['team1', 'team2', 'winner', 'toss_winner']:
    matches[col] = matches[col].replace(team_rename)

# ── deliveries.csv ────────────────────────────────────────────────────────────
deliveries['dismissal_kind']   = deliveries['dismissal_kind'].fillna('not_out')
deliveries['player_dismissed'] = deliveries['player_dismissed'].fillna('N/A')
deliveries['fielder']          = deliveries['fielder'].fillna('N/A')
deliveries['extras_type']      = deliveries['extras_type'].fillna('none')

print('\nAfter cleaning (matches):')
print(matches.isnull().sum())
print('\nAfter cleaning (deliveries):')
print(deliveries.isnull().sum())

## 4. NumPy Operations

In [ ]:
total_runs    = deliveries['total_runs'].values
batsman_runs  = deliveries['batsman_runs'].values   # column name in this dataset
extra_runs    = deliveries['extra_runs'].values

print('== NumPy Runs Statistics (Ball-by-Ball) ==')
print(f'Total Runs Scored   : {np.sum(total_runs):,}')
print(f'Mean Runs / Ball    : {np.mean(total_runs):.3f}')
print(f'Median Runs / Ball  : {np.median(total_runs):.2f}')
print(f'Std Deviation       : {np.std(total_runs):.3f}')
print(f'Min Runs / Ball     : {np.min(total_runs)}')
print(f'Max Runs / Ball     : {np.max(total_runs)}')
print(f'25th Percentile     : {np.percentile(total_runs, 25):.2f}')
print(f'75th Percentile     : {np.percentile(total_runs, 75):.2f}')

# Verify: batsman_runs + extra_runs == total_runs
recalc = np.add(batsman_runs, extra_runs)
print(f'\nVerification — recalculated total : {np.sum(recalc):,}')
print(f'Original total_runs column        : {np.sum(total_runs):,}')

# Boundary (4 or 6) percentage of all balls
boundary_mask = np.isin(batsman_runs, [4, 6])
print(f'\nBoundary ball %  : {np.mean(boundary_mask)*100:.2f}%')

# Dot ball percentage
dot_mask = batsman_runs == 0
print(f'Dot ball %       : {np.mean(dot_mask)*100:.2f}%')

# Log-transform for skewed run distribution
log_runs = np.log1p(batsman_runs)
print(f'\nLog-transformed runs — Mean: {np.mean(log_runs):.3f}, Std: {np.std(log_runs):.3f}')

## 5. Feature Engineering

In [ ]:
# ── Match-level features ──────────────────────────────────────────────────────

# 1. Did the toss winner also win the match?
matches['toss_win_match_win'] = matches['toss_winner'] == matches['winner']

# 2. Result type — uses `result` column (values: 'runs', 'wickets', 'tie', 'no result')
result_map = {
    'runs'      : 'Won by Runs',
    'wickets'   : 'Won by Wickets',
    'tie'       : 'Tie / Super Over',
    'no result' : 'No Result',
}
matches['result_type'] = matches['result'].map(result_map).fillna('Unknown')

# 3. IPL Era
def get_era(year):
    if year <= 2010:
        return 'Early Era (2008-2010)'
    elif year <= 2014:
        return 'Growth Era (2011-2014)'
    elif year <= 2018:
        return 'Peak Era (2015-2018)'
    else:
        return 'Modern Era (2019+)'

matches['era'] = matches['season'].apply(get_era)

# ── Delivery-level features ───────────────────────────────────────────────────
# Note: `over` is 0-indexed (0–19); phase cut-offs shift by -1 vs 1-indexed

# 4. Match phase
def get_phase(over):
    if over <= 5:          # overs 0-5  → Powerplay (balls 1-6 in 1-indexed)
        return 'Powerplay (1-6)'
    elif over <= 14:       # overs 6-14 → Middle
        return 'Middle (7-15)'
    else:                  # overs 15-19 → Death
        return 'Death (16-20)'

deliveries['phase'] = deliveries['over'].apply(get_phase)

# 5. Boundary flag (off bat only)
deliveries['is_boundary'] = deliveries['batsman_runs'].isin([4, 6])

# 6. Dot ball flag (no runs off bat)
deliveries['is_dot'] = deliveries['batsman_runs'] == 0

# 7. Normalised impact score per batter
batter_totals = deliveries.groupby('batter')['batsman_runs'].sum()
deliveries['impact_score'] = np.round(
    deliveries['batter'].map(batter_totals) / batter_totals.max() * 10, 2
)

print('New match-level features:')
print(matches[['id','season','era','result_type','toss_win_match_win']].head(8))
print('\nNew delivery-level features:')
print(deliveries[['match_id','over','phase','batsman_runs','is_boundary','is_dot','impact_score']].head(8))

## 6. Filtering Operations

In [ ]:
# 1. Matches won by runs (batting-first team won)
bat_first_wins = matches[matches['result'] == 'runs']
print(f'Matches won by runs (bat first)  : {len(bat_first_wins)}')

# 2. Dominant wins — margin > 50 runs or 8+ wickets
big_wins = matches[
    ((matches['result'] == 'runs')    & (matches['result_margin'] > 50)) |
    ((matches['result'] == 'wickets') & (matches['result_margin'] >= 8))
]
print(f'Dominant wins (>50R or 8+W)      : {len(big_wins)}')

# 3. Super Over matches
super_overs = matches[matches['super_over'] == 'Y']
print(f'Super Over matches               : {len(super_overs)}')

# 4. Death-over sixes (over 15-19, 0-indexed)
death_sixes = deliveries[
    (deliveries['phase'] == 'Death (16-20)') &
    (deliveries['batsman_runs'] == 6)
]
print(f'Death-over sixes                 : {len(death_sixes)}')

# 5. Matches where toss winner lost
toss_lost = matches[
    (~matches['toss_win_match_win']) &
    (matches['result'] != 'no result')
]
print(f'Matches where toss winner lost   : {len(toss_lost)}')

# 6. Deliveries by well-known spinners (regex on bowler name)
spin_pattern = r'Chahal|Jadeja|Ashwin|Kuldeep|Narine|Tahir|Mishra'
spin_deliveries = deliveries[deliveries['bowler'].str.contains(spin_pattern, na=False)]
print(f'Deliveries by noted spinners     : {len(spin_deliveries)}')

## 7. GroupBy Operations

In [ ]:
# 1. Top 10 run-scorers  (batter column in this dataset)
batter_stats = deliveries.groupby('batter').agg(
    Total_Runs  = ('batsman_runs', 'sum'),
    Balls_Faced = ('batsman_runs', 'count'),
    Boundaries  = ('is_boundary',  'sum'),
    Dot_Balls   = ('is_dot',       'sum')
)
batter_stats['Strike_Rate'] = np.round(
    batter_stats['Total_Runs'] / batter_stats['Balls_Faced'] * 100, 2
)
top_batters = batter_stats.sort_values('Total_Runs', ascending=False).head(10)
print('Top 10 Run Scorers:')
print(top_batters)

In [ ]:
# 2. Top 10 wicket-takers
# Use is_wicket == 1 AND bowler-credited dismissal types
bowler_credit = ['caught', 'bowled', 'lbw', 'stumped', 'caught and bowled', 'hit wicket']
wickets_df = deliveries[
    (deliveries['is_wicket'] == 1) &
    (deliveries['dismissal_kind'].isin(bowler_credit))
]

bowler_stats = wickets_df.groupby('bowler').agg(
    Wickets=('player_dismissed', 'count')
)
bowler_runs  = deliveries.groupby('bowler')['total_runs'].sum()
bowler_balls = deliveries.groupby('bowler')['total_runs'].count()
bowler_stats['Economy'] = np.round(bowler_runs / bowler_balls * 6, 2)

top_bowlers = bowler_stats.sort_values('Wickets', ascending=False).head(10)
print('Top 10 Wicket Takers:')
print(top_bowlers)

In [ ]:
# 3. Team-wise win count
team_wins = (
    matches[matches['result'] != 'no result']
    .groupby('winner')
    .agg(Wins=('id', 'count'))
    .sort_values('Wins', ascending=False)
)
print('Team-wise Win Counts:')
print(team_wins)

In [ ]:
# 4. Season-wise stats
# matches.id links to deliveries.match_id
season_matches = matches.groupby('season').agg(
    Total_Matches=('id', 'count')
)

season_runs = (
    deliveries
    .merge(matches[['id', 'season']], left_on='match_id', right_on='id', how='left')
    .groupby('season')['total_runs']
    .sum()
    .rename('Total_Runs')
)

season_stats = season_matches.join(season_runs)
season_stats['Avg_Runs_Per_Match'] = np.round(
    season_stats['Total_Runs'] / season_stats['Total_Matches'], 2
)
print('Season-wise Stats:')
print(season_stats)

## 8. Melt / Reshape

In [ ]:
# Phase-wise runs for top 5 teams
# batting_team is already a column in deliveries — no merge needed
top5_teams = team_wins.head(5).index.tolist()

phase_runs = (
    deliveries[deliveries['batting_team'].isin(top5_teams)]
    .groupby(['batting_team', 'phase'])['batsman_runs']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure all three phase columns exist
for col in ['Powerplay (1-6)', 'Middle (7-15)', 'Death (16-20)']:
    if col not in phase_runs.columns:
        phase_runs[col] = 0

melted_df = pd.melt(
    phase_runs,
    id_vars    = ['batting_team'],
    value_vars = ['Powerplay (1-6)', 'Middle (7-15)', 'Death (16-20)'],
    var_name   = 'Phase',
    value_name = 'Runs'
)

print(f'Original shape : {phase_runs.shape}')
print(f'Melted shape   : {melted_df.shape}')
print('\nMelted sample:')
display(melted_df.head(12))

## 9. Data Visualization

In [ ]:
plt.figure(figsize=(18, 12))

# 1. Top 10 Run Scorers
plt.subplot(2, 3, 1)
top10_bat = batter_stats.nlargest(10, 'Total_Runs')
plt.barh(top10_bat.index, top10_bat['Total_Runs'], color='steelblue')
plt.title('Top 10 IPL Run Scorers')
plt.xlabel('Total Runs')
plt.gca().invert_yaxis()

# 2. Team Wins Pie
plt.subplot(2, 3, 2)
top8_wins = team_wins.head(8)
plt.pie(top8_wins['Wins'], labels=top8_wins.index, autopct='%1.1f%%', startangle=140)
plt.title('Win Share — Top 8 Teams')

# 3. Runs by Phase
plt.subplot(2, 3, 3)
phase_totals = (
    deliveries.groupby('phase')['batsman_runs']
    .sum()
    .reindex(['Powerplay (1-6)', 'Middle (7-15)', 'Death (16-20)'])
)
plt.bar(phase_totals.index, phase_totals.values,
        color=['#4C72B0', '#DD8452', '#55A868'])
plt.title('Total Runs by Match Phase')
plt.ylabel('Runs')
plt.xticks(rotation=15)

# 4. Matches per Season
plt.subplot(2, 3, 4)
season_matches['Total_Matches'].plot(kind='bar', color='coral')
plt.title('Matches per IPL Season')
plt.xlabel('Season')
plt.ylabel('Matches')
plt.xticks(rotation=45)

# 5. Dismissal Type Distribution
plt.subplot(2, 3, 5)
dismissal_counts = (
    deliveries[deliveries['is_wicket'] == 1]
    ['dismissal_kind'].value_counts()
)
dismissal_counts.plot(kind='bar', color='mediumseagreen')
plt.title('Dismissal Types')
plt.xticks(rotation=30)

# 6. Runs per Ball Distribution
plt.subplot(2, 3, 6)
plt.hist(deliveries['batsman_runs'], bins=[0,1,2,3,4,5,6,7],
         color='mediumpurple', alpha=0.8, rwidth=0.85)
plt.title('Runs per Ball Distribution')
plt.xlabel('Runs off Bat')
plt.ylabel('Frequency')
plt.xticks([0,1,2,3,4,5,6])

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap — phase-wise runs per match
corr_data = (
    deliveries
    .groupby(['match_id', 'phase'])['batsman_runs']
    .sum()
    .unstack(fill_value=0)
)
corr_data['Total_Runs'] = corr_data.sum(axis=1)

plt.figure(figsize=(7, 5))
sns.heatmap(corr_data.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Phase-wise Runs Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Stacked bar — phase-wise runs for Top 5 Teams
pivot = melted_df.pivot(index='batting_team', columns='Phase', values='Runs')
pivot = pivot[['Powerplay (1-6)', 'Middle (7-15)', 'Death (16-20)']]

pivot.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='Set2')
plt.title('Phase-wise Runs — Top 5 Teams')
plt.xlabel('Team')
plt.ylabel('Total Runs')
plt.xticks(rotation=20)
plt.legend(title='Phase')
plt.tight_layout()
plt.show()

## 10. Final Summary

In [ ]:
print('========== IPL CRICKET STATS ANALYSIS — SUMMARY ==========')
print(f'Total Matches Analysed       : {len(matches)}')
print(f'Total Balls Bowled           : {len(deliveries):,}')
print(f'Total Runs Scored            : {deliveries["total_runs"].sum():,}')
print(f'Total Wickets Fallen         : {deliveries["is_wicket"].sum():,}')
print(f'Highest Run Scorer           : {batter_stats["Total_Runs"].idxmax()} '
      f'({int(batter_stats["Total_Runs"].max())} runs)')
print(f'Highest Wicket Taker         : {top_bowlers["Wickets"].idxmax()} '
      f'({int(top_bowlers["Wickets"].max())} wickets)')
print(f'Most Successful Team         : {team_wins["Wins"].idxmax()} '
      f'({int(team_wins["Wins"].max())} wins)')
print(f'Super Over Matches           : {len(super_overs)}')
print(f'Toss-Win -> Match-Win %      : {matches["toss_win_match_win"].mean()*100:.1f}%')
print(f'Boundary Ball %              : {deliveries["is_boundary"].mean()*100:.2f}%')
print(f'Dot Ball %                   : {deliveries["is_dot"].mean()*100:.2f}%')
print(f'Average Runs per Match       : {season_stats["Avg_Runs_Per_Match"].mean():.2f}')
print('===========================================================')